# Part 23: AutoGen — Multi-Agent Conversations

> Microsoft's AutoGen framework for building multi-agent systems with conversational patterns, tool use, and model-agnostic design.

---


## 23.1 AutoGen Core Concepts

AutoGen models agents as **conversational participants**:

| Concept | Description |
|---------|-------------|
| **AssistantAgent** | LLM-powered agent that responds to messages |
| **UserProxyAgent** | Represents human or executes code locally |
| **GroupChat** | Multi-agent round-robin or custom conversation |
| **ConversableAgent** | Base class for all agents |

```
AutoGen Architecture:
  UserProxy ←→ AssistantAgent
  UserProxy ←→ GroupChat ←→ [Agent1, Agent2, Agent3]
```


In [ ]:
# pip install autogen-agentchat autogen-ext[openai,ollama]
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.ui import Console
import asyncio


## 23.2 Model Clients

In [ ]:
# OpenAI model client
openai_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key="YOUR_OPENAI_KEY"  # or use env var OPENAI_API_KEY
)

# Ollama (local models) — free, private
from autogen_ext.models.ollama import OllamaChatCompletionClient

ollama_client = OllamaChatCompletionClient(
    model="llama3.2",
    host="http://localhost:11434"  # default Ollama server
)

# Anthropic via OpenAI-compatible wrapper
anthropic_client = OpenAIChatCompletionClient(
    model="claude-3-5-haiku-20241022",
    base_url="https://api.anthropic.com/v1",
    api_key="YOUR_ANTHROPIC_KEY"
)


## 23.3 Basic AssistantAgent

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken

agent = AssistantAgent(
    name="helpful_assistant",
    model_client=openai_client,
    system_message="You are a helpful AI assistant. Be concise and accurate."
)

async def run_basic_agent():
    response = await agent.on_messages(
        [TextMessage(content="Explain RAG in 2 sentences.", source="user")],
        cancellation_token=CancellationToken()
    )
    print(response.chat_message.content)

# asyncio.run(run_basic_agent())


## 23.4 Agents with Tools

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_core.tools import FunctionTool
import sqlite3

# Define tool functions
def lookup_product_price(product_name: str) -> str:
    """Look up the price of a product from the database."""
    # Example: SQLite lookup
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE products (name TEXT, price REAL)")
    conn.execute("INSERT INTO products VALUES ('laptop', 999.99)")
    conn.execute("INSERT INTO products VALUES ('mouse', 29.99)")
    conn.commit()
    
    cursor = conn.execute(
        "SELECT price FROM products WHERE name LIKE ?",
        (f"%{product_name.lower()}%",)
    )
    row = cursor.fetchone()
    return f"${row[0]:.2f}" if row else "Product not found"

def calculate_discount(original_price: float, discount_percent: float) -> str:
    """Calculate discounted price."""
    discount = original_price * discount_percent / 100
    final_price = original_price - discount
    return f"Original: ${original_price:.2f} | Discount: ${discount:.2f} | Final: ${final_price:.2f}"

# Wrap as AutoGen tools
price_tool = FunctionTool(
    lookup_product_price,
    description="Look up product prices from the catalog"
)
discount_tool = FunctionTool(
    calculate_discount,
    description="Calculate discounted price given original price and discount percentage"
)

# Agent with tools + reflect_on_tool_use
pricing_agent = AssistantAgent(
    name="pricing_agent",
    model_client=openai_client,
    tools=[price_tool, discount_tool],
    system_message="You are a pricing assistant. Use tools to answer price-related questions.",
    reflect_on_tool_use=True  # agent reflects on tool output before final response
)


## 23.5 Multi-Agent Team (RoundRobinGroupChat)

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console

# Define specialist agents
data_analyst = AssistantAgent(
    name="data_analyst",
    model_client=openai_client,
    system_message="You are a data analyst. Analyze data patterns and provide insights."
)

ml_engineer = AssistantAgent(
    name="ml_engineer",
    model_client=openai_client,
    system_message="You are an ML engineer. Suggest appropriate models and architectures."
)

writer = AssistantAgent(
    name="technical_writer",
    model_client=openai_client,
    system_message="You are a technical writer. Summarize discussions into clear documentation. When done, say TERMINATE."
)

# Termination conditions
termination = (
    MaxMessageTermination(max_messages=10) |
    TextMentionTermination("TERMINATE")
)

# Create team
team = RoundRobinGroupChat(
    participants=[data_analyst, ml_engineer, writer],
    termination_condition=termination
)

async def run_team():
    task = "Design a machine learning pipeline to predict customer churn from transaction data."
    await Console(team.run_stream(task=task))

# asyncio.run(run_team())


## 23.6 SelectorGroupChat (Smart Routing)

In [ ]:
from autogen_agentchat.teams import SelectorGroupChat

# SelectorGroupChat uses an LLM to decide who speaks next
smart_team = SelectorGroupChat(
    participants=[data_analyst, ml_engineer, writer],
    model_client=openai_client,
    termination_condition=termination,
    selector_prompt="""You are managing a conversation between specialists.
Given the conversation history, select who should speak next.
Participants: {participants}
Conversation: {history}
Select the most appropriate participant to continue.""",
    allow_repeated_speaker=False  # prevent same agent speaking twice in a row
)

# Use case: customer support with routing
customer_service = AssistantAgent(
    name="customer_service",
    model_client=openai_client,
    system_message="You handle general customer inquiries and route to specialists."
)

tech_support = AssistantAgent(
    name="tech_support",
    model_client=openai_client,
    system_message="You solve technical problems and bugs."
)

billing_support = AssistantAgent(
    name="billing_support",
    model_client=openai_client,
    system_message="You handle billing, refunds, and subscription questions."
)

support_team = SelectorGroupChat(
    participants=[customer_service, tech_support, billing_support],
    model_client=openai_client,
    termination_condition=TextMentionTermination("RESOLVED")
)


## 23.7 Nested Teams & Hierarchical Agents

In [ ]:
# An orchestrator that delegates to sub-teams

orchestrator = AssistantAgent(
    name="orchestrator",
    model_client=openai_client,
    system_message="""You are a project orchestrator. 
Break down complex projects and coordinate specialists.
You have access to: research_team, development_team, review_team."""
)

# Each sub-team is itself a multi-agent group
research_team = RoundRobinGroupChat(
    participants=[
        AssistantAgent(name="web_researcher",    model_client=openai_client, system_message="You research web sources."),
        AssistantAgent(name="data_researcher",   model_client=openai_client, system_message="You analyze datasets."),
    ],
    termination_condition=MaxMessageTermination(6)
)

async def hierarchical_workflow(project: str):
    # Phase 1: Research
    print("=== Research Phase ===")
    research_result = await Console(research_team.run_stream(task=f"Research: {project}"))
    
    # Phase 2: Use research results in next phase
    # (In production: extract and pass structured results)
    print("\n=== Orchestration Phase ===")
    final = await orchestrator.on_messages(
        [TextMessage(content=f"Project: {project}\nResearch done. Create final summary.", source="user")],
        cancellation_token=CancellationToken()
    )
    return final.chat_message.content


## 23.8 Model-Agnostic Design

In [ ]:
# Swap model backends without changing agent logic

def create_agent_with_model(model_backend: str = "openai") -> AssistantAgent:
    if model_backend == "openai":
        client = OpenAIChatCompletionClient(model="gpt-4o-mini")
    elif model_backend == "ollama":
        client = OllamaChatCompletionClient(model="llama3.2")
    elif model_backend == "anthropic":
        client = OpenAIChatCompletionClient(
            model="claude-3-5-haiku-20241022",
            base_url="https://api.anthropic.com/v1",
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
    
    return AssistantAgent(
        name="flexible_agent",
        model_client=client,
        system_message="You are a helpful assistant."
    )

# Use case: cost optimization
# Development → Ollama (free)
# Testing     → GPT-4o-mini (cheap)  
# Production  → GPT-4o or Claude (quality)


## 23.9 Summary

| Feature | AutoGen API |
|---------|------------|
| Basic agent | `AssistantAgent(name, model_client, system_message)` |
| With tools | `tools=[FunctionTool(fn, description)]` |
| Tool reflection | `reflect_on_tool_use=True` |
| Round-robin team | `RoundRobinGroupChat(participants, termination)` |
| Smart routing | `SelectorGroupChat(participants, model_client)` |
| Termination | `MaxMessageTermination(n)` or `TextMentionTermination("word")` |
| Stream output | `Console(team.run_stream(task=...))` |
| Cancellation | `CancellationToken()` |

---

**Next:** [Part 24 — Model Context Protocol (MCP)](Part24_Model_Context_Protocol.ipynb)
